In [5]:
import scipy.io
import numpy as np

import sys
sys.path.insert(0, '../src')      # notebooks/ -> project root -> src/
from matio import peek, load_mat


# # Load
# data = scipy.io.loadmat('../data/meg_mri/RSID0001_meg_rest_60sec.mat')

# # See what's inside
# print(data.keys())

# trial = data['trial']
# time = data['time']
# fsample = data['fsample']

# meg_data = trial[0, 0]
# time_vec = time[0, 0]
# fs = fsample[0, 0]

# print('MEG data shape:', meg_data.shape)
# print('Sampling rate:', fs)
# print('Duration:', meg_data.shape[1] / fs)

# print("data/ts48/ts48AD : ")

d = load_mat('../data/ts48/ts48AD.mat')
m = d['matrix']          # (77, 48, 36000) — subjects, regions, time
m[0, 0, :2000]           # subject 0, region 0, first ~3 s
print(d.keys())



dict_keys(['matrix', 'radid', 'labels'])


In [ ]:
import scipy.io
import numpy as np

data = scipy.io.loadmat('../data/meg_mri/RSID0001_meg_rest_60sec.mat')
grad = data['grad']

print(data.keys())

# See what fields are inside grad
print('grad fields:', grad.dtype.names)

chanpos = grad[0, 0]['chanpos']
chanori = grad[0, 0]['chanori']
print('chanpos shape:', chanpos.shape)
print('chanori shape:', chanori.shape)
# print('chanpos first row:', chanpos[0])

print(data['grad'][0][0]['chanpos'])

dict_keys(['__header__', '__version__', '__globals__', 'hdr', 'fsample', 'sampleinfo', 'grad', 'elec', 'trial', 'time', 'label', 'cfg'])
grad fields: ('balance', 'chanori', 'chanpos', 'chantype', 'chanunit', 'coilori', 'coilpos', 'coordsys', 'label', 'tra', 'type', 'unit')
chanpos shape: (301, 3)
chanori shape: (301, 3)
[[ 9.06967744e+00  1.74215249e+00  1.58876392e+01]
 [ 7.87415855e+00  3.48680858e+00  1.63567890e+01]
 [ 6.86851022e+00  5.40463571e+00  1.62151885e+01]
 [ 5.37949586e+00  7.21259831e+00  1.58619254e+01]
 [ 3.79327515e+00  8.55920973e+00  1.53904069e+01]
 [ 1.88727243e+00  9.47371392e+00  1.47998792e+01]
 [-1.99675560e-01  9.77102582e+00  1.37782951e+01]
 [ 5.95018729e+00  3.64771379e+00  1.73709672e+01]
 [ 4.91454235e+00  5.54548437e+00  1.71676455e+01]
 [ 3.43177580e+00  7.07728888e+00  1.68680459e+01]
 [ 1.63306209e+00  8.16729521e+00  1.66179425e+01]
 [-4.02132175e-01  8.84119671e+00  1.57536305e+01]
 [ 1.76575110e+00  6.21736716e+00  1.79780254e+01]
 [-4.38161039e-

In [ ]:
import os

data_dir = '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/meg_mri'

files = []
for f in os.listdir(data_dir):
    if f.endswith('_meg_rest_60sec.mat'):
        files.append(f)

labels = {}
for f in files:
    subject_id = f.split('_')[0]          # e.g. 'RSID0001'
    number = int(subject_id[4:])          # e.g. 1
    label = 0 if number <= 70 else 1      # 0 = control, 1 = AD
    labels[subject_id] = label

print(f"Total subjects: {len(labels)}")
print(f"Controls: {sum(v == 0 for v in labels.values())}")
print(f"AD: {sum(v == 1 for v in labels.values())}")


grad = data['grad']

grad_labels = [str(grad[0,0]['label'][i, 0][0]) for i in range(grad[0,0]['label'].shape[0])]
meg_labels = [str(data['label'][i, 0][0]) for i in range(data['label'].shape[0])]

print('grad channels:', len(grad_labels))
print('meg channels:', len(meg_labels))
print('first 5 grad labels:', grad_labels[:5])
print('first 5 meg labels:', meg_labels[:5])

# Find which grad indices match our 271 meg channels
matching_indices = [grad_labels.index(lbl) for lbl in meg_labels]
print('matching indices found:', len(matching_indices))


Total subjects: 148
Controls: 70
AD: 78
grad channels: 301
meg channels: 271
first 5 grad labels: ['MLC11', 'MLC12', 'MLC13', 'MLC14', 'MLC15']
first 5 meg labels: ['MLC11', 'MLC12', 'MLC13', 'MLC14', 'MLC15']
matching indices found: 271


In [12]:
import nibabel as nib
import numpy as np

img = nib.load('../data/brainnetome_atlas/BN_Atlas_246_2mm.nii.gz')
data = img.get_fdata()      # the 3D array
print(data.shape)           # roughly (91, 109, 91) at 2mm
ids, counts = np.unique(data, return_counts=True)

centroids = {}
for rid in range(1, 247):
    voxels = np.argwhere(data == rid)
    if len(voxels) == 0:
        print(f'warning: region {rid} has no voxels')
        continue
    centroids[rid] = nib.affines.apply_affine(img.affine, voxels.mean(axis=0))

print(len(centroids), 'centroids computed')
print('region 1:', centroids[1])
print('region 2:', centroids[2])



(91, 109, 91)
246 centroids computed
region 1: [-4.72060683 15.53476612 53.70164349]
region 2: [ 6.95583239 16.49830125 54.22197055]
